# Parcours complet du RAG ENSEA

Point d'entrée du projet. Ce notebook présente toutes les étapes, leurs entrées, leurs sorties et l'ordre recommandé.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


## Architecture suivie

`Sites ENSEA + Facebook + bibliothèque PDF économique → extraction de faits tabulaires → nettoyage → chunks narratifs et atomiques → embeddings → LanceDB + BM25 → retrieval hybride → reranking → génération citée → évaluation → observabilité`

Le dossier `parcours_rag/` contient le chemin principal. Les notebooks numérotés directement dans `notebooks/` restent les expériences historiques qui ont servi à sélectionner les paramètres.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


In [ ]:
import pandas as pd
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
stages = [
    (1, 'Collecte institutionnelle', 'HTML, PDF', 'data/raw, data/pdf'),
    (2, 'Collecte Facebook', 'Pages clubs', 'data/facebook/raw'),
    (3, 'Import PDF économiques', 'PDF + tableaux', 'texte + faits indicateur/période/valeur/page'),
    (4, 'Nettoyage', 'Documents + faits', 'Markdown + faits tabulaires conservés'),
    (5, 'Chunking', 'Markdown + faits', 'chunks text et table_fact séparés par route'),
    (6, 'Embeddings', 'Chunks enfants', 'trois matrices BGE-M3'),
    (7, 'Indexation', 'Chunks + vecteurs', 'trois couples LanceDB + BM25'),
    (8, 'Retrieval', 'Question', 'passages rerankés'),
    (9, 'Génération', 'Question + passages', 'réponse + citations'),
    (10, 'Évaluation', 'Golden set', 'Recall, MRR, page, relation valeur–année, latence'),
    (11, 'Observabilité', 'Traces', 'cache, erreurs, P95, routes'),
    (12, 'Mises à jour', 'PDF ajouté', 'index incrémental'),
    (13, 'RAG complet', 'Tous les composants', 'assistant utilisable'),
]
display(pd.DataFrame(stages, columns=['étape', 'composant', 'entrée', 'sortie']))


In [ ]:
artifacts = {
    'documents_bruts': config.project.data_path / 'raw',
    'bibliothèque_eco_rci': config.local_documents.source_full_path,
    'documents_nettoyés': config.project.data_path / 'cleaned',
    'chunks': config.project.data_path / 'chunks' / 'chunks.jsonl',
    'lancedb': config.vectorstore.uri_path,
    'bm25': config.vectorstore.bm25_index_full_path,
    'chunks_économiques': config.local_documents.chunks_full_path,
    'lancedb_économique': PROJECT_ROOT / config.local_documents.vectorstore_uri,
    'bm25_économique': PROJECT_ROOT / config.local_documents.bm25_index_path,
    'golden_set': config.evaluation.expanded_golden_set_full_path,
    'traces': config.observability.traces_full_path,
}
display(pd.DataFrame([
    {'artefact': name, 'chemin': str(path), 'présent': path.exists()}
    for name, path in artifacts.items()
]))


## Ordre d'exécution

Arrêtez l'application FastAPI et fermez les autres kernels avant de commencer. N'essayez pas d'utiliser l'assistant avant la fin du notebook 07 : les anciens index ont été volontairement supprimés.

1. Exécutez les notebooks 01 à 07 pour reconstruire les corpus et les trois index.
2. Les notebooks 08 et 09 expliquent le chemin d'une question jusqu'à la réponse.
3. Les notebooks 10 et 11 mesurent la qualité et le comportement du système.
4. Les notebooks 12 et 13 couvrent l'exploitation quotidienne et le pipeline final.

Les notebooks 02 à 07 contiennent les opérations actives de reconstruction. Laissez chaque opération se terminer et sauvegardez le notebook avant de passer au suivant. Le notebook 06 charge BGE-M3 sur le GPU : aucun autre kernel ne doit occuper la VRAM.
